## Импорты

Подключение модулей для работы с CSV, JSON, регулярными выражениями, датой и путями к файлам.

In [ ]:
import csv
import json
import re
from datetime import datetime
from pathlib import Path

## Базовый класс `Account`

Реализация банковского счёта, основных операций, истории транзакций, загрузки и очистки данных.

In [ ]:
class Account:
    """Базовая модель банковского счёта с историей операций."""

    _account_counter = 1000

    def __init__(self, account_holder: str, balance: float = 0):
        name_pattern = (
            r"(?:[A-Z][a-z]* [A-Z][a-z]*|"
            r"[А-ЯЁ][а-яё]* [А-ЯЁ][а-яё]*)"
        )
        if (
            not isinstance(account_holder, str)
            or not re.fullmatch(name_pattern, account_holder)
        ):
            raise ValueError(
                "Имя владельца должно быть в формате 'Имя Фамилия'"
            )
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self.holder = account_holder
        self.account_number = f"ACC-{Account._account_counter:04d}"
        Account._account_counter += 1
        self._balance = balance
        self.operations_history = []

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной")

        self._balance += amount
        self.operations_history.append(
            {
                "type": "deposit",
                "amount": amount,
                "datetime": datetime.now(),
                "balance": self._balance,
                "status": "success",
            }
        )

    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной")

        if amount <= self._balance:
            self._balance -= amount
            status = "success"
        else:
            status = "fail"

        self.operations_history.append(
            {
                "type": "withdraw",
                "amount": amount,
                "datetime": datetime.now(),
                "balance": self._balance,
                "status": status,
            }
        )

    def get_balance(self):
        return self._balance

    def get_history(self):
        return self.operations_history

    def get_large_transactions(self, n):
        return sorted(
            self.operations_history,
            key=lambda operation: (operation["amount"], operation["datetime"]),
            reverse=True,
        )[:n]

    def load_history(self, filename, account_number=None):
        """Загрузить из CSV/JSON операции выбранного счёта."""
        path = Path(filename)
        text = path.read_text(encoding="utf-8")

        # Удаляем служебные номера строк, встречающиеся в исходных файлах.
        text = re.sub(r"(?m)^\s*\d+\|", "", text)

        if path.suffix.lower() == ".csv":
            transactions = list(csv.DictReader(text.splitlines()))
        elif path.suffix.lower() == ".json":
            transactions = json.loads(text)
        else:
            raise ValueError("Поддерживаются только CSV и JSON файлы")

        target_account_number = account_number or self.account_number
        self.operations_history.extend(
            transaction
            for transaction in transactions
            if transaction.get("account_number") == target_account_number
            and transaction.get("account_type") == self.account_type
        )
        self.clean_history()

        if account_number is not None:
            self.account_number = account_number
        if self.operations_history:
            self._balance = self.operations_history[-1]["balance"]

    def clean_history(self):
        """Удалить операции с некорректными полями."""
        valid_operation_types = {"deposit", "withdraw"}
        if self.account_type == "savings":
            valid_operation_types.add("interest")

        clean_operations = []
        for operation in self.operations_history:
            operation_type = operation.get("operation", operation.get("type"))
            amount = operation.get("amount")
            date = operation.get("date", operation.get("datetime"))
            balance = operation.get("balance_after", operation.get("balance"))
            status = operation.get("status")

            try:
                amount = float(amount)
                balance = float(balance)
                if isinstance(date, str):
                    date = datetime.strptime(date, "%Y-%m-%d %H:%M:%S")
            except (TypeError, ValueError):
                continue

            if (
                operation_type not in valid_operation_types
                or amount <= 0
                or not isinstance(date, datetime)
                or status not in {"success", "fail"}
            ):
                continue

            clean_operations.append(
                {
                    "type": operation_type,
                    "amount": amount,
                    "datetime": date,
                    "balance": balance,
                    "status": status,
                }
            )

        self.operations_history = sorted(
            clean_operations, key=lambda operation: operation["datetime"]
        )

## Класс `CheckingAccount`

Расчётный счёт наследует базовое поведение `Account` и задаёт тип счёта `checking`.

In [ ]:
class CheckingAccount(Account):
    """Расчётный счёт."""

    account_type = "checking"

## Класс `SavingsAccount`

Сберегательный счёт поддерживает начисление процентов и ограничивает снятие половиной текущего баланса.

In [ ]:
class SavingsAccount(Account):
    """Сберегательный счёт с начислением процентов."""

    account_type = "savings"

    def apply_interest(self, rate):
        if rate <= 0:
            raise ValueError("Процентная ставка должна быть положительной")

        interest = self._balance * rate / 100
        self._balance += interest
        self.operations_history.append(
            {
                "type": "interest",
                "amount": interest,
                "datetime": datetime.now(),
                "balance": self._balance,
                "status": "success",
            }
        )

    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной")

        if amount > self._balance * 0.5:
            self.operations_history.append(
                {
                    "type": "withdraw",
                    "amount": amount,
                    "datetime": datetime.now(),
                    "balance": self._balance,
                    "status": "fail",
                }
            )
            return

        super().withdraw(amount)